# EJS Partials

Partials are reusable fragments of template code — a header, a footer, a navigation bar, a product card — that you include inside multiple pages instead of copy-pasting the same markup everywhere. You render them with the **unescaped** output tag:

```html
<%- include('partials/header') %>
```

The payoff is one edit instead of many: change the nav in one file and every page updates.

---

## How to Use Partials

### 1. Create a folder

```
views/
├── partials/
│   ├── header.ejs
│   └── footer.ejs
├── home.ejs
└── about.ejs
```

The folder name `partials` is convention, not a requirement — EJS doesn't treat it specially.

### 2. Write the partial

**`views/partials/header.ejs`**
```html
<nav>
  <a href="/">Home</a> |
  <a href="/about">About</a>
</nav>
```

A partial is just an `.ejs` file. There's no special syntax marking it as a partial.

### 3. Include it

```html
<body>
  <%- include('partials/header') %>
  <h1>Main Content</h1>
  <%- include('partials/footer') %>
</body>
```

### 4. Pass local data

Variables from the parent are automatically visible inside the partial. Pass a second argument to add or override:

```html
<%- include('partials/header', { title: 'My Page' }) %>
```

---

## Key Rules

### Path resolution is relative to the *calling file*

`include('partials/header')` resolves relative to the file it's written in, **not** the views root. From `views/users/profile.ejs`, that looks for `views/users/partials/header.ejs` — which doesn't exist.

Two fixes:

```html
<%- include('../partials/header') %>   <!-- relative, brittle -->
<%- include('/partials/header') %>     <!-- anchored at views root -->
```

**Prefer the leading slash.** It works identically from every file regardless of nesting depth, so moving a template into a subfolder doesn't break it.

### Always use `<%-`, never `<%=`

```html
<%- include('partials/header') %>   <!-- renders the HTML -->
<%= include('partials/header') %>   <!-- prints &lt;nav&gt;&lt;a href... as visible text -->
```

`<%=` escapes its output. Since a partial *returns HTML*, escaping it turns your nav bar into literal angle-bracket text on the page. If you see raw tags rendered on screen, this is why.

This is the one place the unescaped tag is unambiguously correct — the content is yours, not user input.

### The extension is optional

`include('partials/header')` and `include('partials/header.ejs')` both work. Omitting it is conventional.

---

## Variable Scope

A partial can see everything the parent template can see:

```js
res.render('home', { user: { name: 'Alex' } });
```

```html
<!-- home.ejs -->
<%- include('partials/header') %>
```

```html
<!-- partials/header.ejs — user is available here without being passed -->
<% if (user) { %><span>Hi, <%= user.name %></span><% } %>
```

This is convenient but creates hidden coupling: the partial silently depends on a variable the parent happened to have. If someone renders a page without `user`, the partial throws `user is not defined`.

**Two ways to make partials safe:**

```html
<!-- 1. Defensive check inside the partial -->
<% if (locals.user) { %><span>Hi, <%= user.name %></span><% } %>

<!-- 2. Explicit contract — pass everything the partial needs -->
<%- include('partials/header', { user: locals.user || null }) %>
```

Variables passed as the second argument are **scoped to that partial only** — they don't leak back into the parent or into sibling includes.

For things every page needs (site name, logged-in user, current year), `app.locals` and `res.locals` are cleaner than threading them through every include:

```js
app.locals.siteName = 'My Store';
app.use((req, res, next) => {
  res.locals.currentUser = req.user || null;
  next();
});
```

---

## Partials in Loops

The most useful pattern — one card template, many items:

```html
<div class="row">
  <% products.forEach(product => { %>
    <%- include('/partials/product-card', { product }) %>
  <% }); %>
</div>
```

**`views/partials/product-card.ejs`**
```html
<article class="card">
  <h3><%= product.name %></h3>
  <p><%= product.description %></p>
  <span>₹<%= product.price %></span>
</article>
```

Note `{ product }` — shorthand for `{ product: product }`. Each iteration gets its own scope, so the partial always sees the current item.

---

## The Layout Sandwich

EJS has **no built-in layout system**. The standard workaround is splitting the page shell into two partials with deliberately unbalanced tags:

**`views/partials/head.ejs`**
```html
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title><%= typeof title !== 'undefined' ? title : 'My Site' %></title>
  <link rel="stylesheet" href="/css/bootstrap.min.css">
</head>
<body>
  <%- include('/partials/nav') %>
```

**`views/partials/foot.ejs`**
```html
  <footer>&copy; <%= new Date().getFullYear() %></footer>
  <script src="https://code.jquery.com/jquery-3.5.1.slim.min.js"></script>
  <script src="https://cdn.jsdelivr.net/npm/popper.js@1.16.1/dist/umd/popper.min.js"></script>
  <script src="/js/bootstrap.min.js"></script>
</body>
</html>
```

**Every page then becomes:**
```html
<%- include('/partials/head', { title: 'Home' }) %>

<h1>Welcome</h1>

<%- include('/partials/foot') %>
```

Yes, the individual files have unclosed tags — editors will complain and auto-indent gets confused. That's the tradeoff. The alternative is `express-ejs-layouts`, which gives you a real layout file with a `<%- body %>` placeholder:

```bash
npm install express-ejs-layouts
```

```js
const expressLayouts = require('express-ejs-layouts');
app.use(expressLayouts);
app.set('layout', 'layouts/main');
```

---

## Dynamic Partial Paths

The path can be an expression, which is handy for rendering by type:

```html
<%- include(`/partials/alerts/${flash.type}`, { message: flash.message }) %>
```

**Never build this from user input.** `include('/partials/' + req.query.name)` is a path traversal vulnerability. Whitelist instead:

```js
const allowed = ['success', 'error', 'warning'];
const type = allowed.includes(flash.type) ? flash.type : 'info';
```

---

## Naming Conventions

Common approaches — pick one and stay consistent:

| Style | Example |
|---|---|
| Folder-based | `views/partials/header.ejs` |
| Underscore prefix | `views/_header.ejs` |
| Grouped by feature | `views/products/_card.ejs` |

Name by **role**, not position: `product-card.ejs` beats `middle-section.ejs`. Six months later you'll know what the first one does.

---

## Common Errors

| Symptom | Cause |
|---|---|
| Raw `<nav>` tags visible on the page | Used `<%=` instead of `<%-` |
| `Could not find the include file` | Path relative to the calling file — add a leading `/` |
| `x is not defined` | Partial expects a variable the parent didn't have — use `locals.x` |
| Partial renders once, not per item | `include` placed outside the loop body |
| `Unexpected token` after adding a partial | Unbalanced braces in the sandwich partials |
| Changes to a partial don't appear | View cache on (`NODE_ENV=production`), or browser cache |

---

## Notes and Gotchas

- **`include` is a function in EJS 3.** The old EJS 1/2 syntax `<% include header %>` was removed. Tutorials showing it are pre-2017.
- **Includes are resolved at render time**, not compile time — a dynamic path works, but each include is a real file read on a cache miss.
- **Deep nesting costs.** Partials can include other partials, and it works, but a five-level chain makes it hard to trace where a variable came from. Two levels is usually plenty.
- **No circular includes.** A partial including itself (directly or via a chain) will blow the stack.
- **In production, set `NODE_ENV=production`** so compiled templates — partials included — are cached rather than re-read per request.
- **Keep logic out.** If a partial needs five lines of computation before it can render, do that work in the route and pass the result.

---

## Quick Reference

```html
<%- include('/partials/header') %>                        <!-- from views root -->
<%- include('partials/header') %>                         <!-- relative to this file -->
<%- include('/partials/head', { title: 'Home' }) %>       <!-- with data -->
<%- include('/partials/card', { product }) %>             <!-- inside a loop -->
<% if (locals.user) { %> ... <% } %>                      <!-- safe optional var -->
```

Rule of thumb: if you've pasted the same markup into a second file, it's a partial.